# Figure 4 — preamble

Loads the dataset and the Figure-4 GAM export. Nothing is re-fit here.


In [ ]:
import os
import sys
import re
import subprocess
import warnings

subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])
warnings.simplefilter('ignore', UserWarning)
warnings.simplefilter('ignore', FutureWarning)

import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.notebook import load_notebook_data
from src.gams import (grid_derivatives, deriv_map_spatial_null_p, sa_metric_map,
                      bootstrap_metric_sa_corr, sa_decile_change,
                      SIGNAL_TYPES, signal_labels, signal_units, gam_variant_tags,
                      gam_cache_path, build_nuisance)
from src.nulls import (null_basis, generate_surrogates, corr_with_null,
                       require_full_parcellation)
from src.plotting import figure_scratch_dir, setup_notebook_style
from snaplab_tools.plotting.utils import save_panel
from snaplab_tools.stats import pca_with_nan_handling
from snaplab_tools.plotting.plotting import plot_correlation, plot_brain_surface_data_single
from snaplab_tools.plotting.utils import register_custom_colormaps, get_snap_colors

setup_notebook_style()

%matplotlib inline


In [ ]:
dataset        = 'HCP-D'    # 'HCP-D' 'PNC'
which_modality = 'FMRI'
n_regions      = 400

if dataset == 'HCP-D':
    output_type = 'timescale_auc_zero_crossing_bp-0.01-0.08hz'
elif dataset == 'PNC':
    output_type = 'timescale_auc_zero_crossing'

# ── Brain-gene panel ────────────────────────────────────────────────────────────────────────────
# Which gene list the gene PCA is fit on. Resolves to data/supporting_data/<tag>_brain_genes.txt,
# so a new list needs no code change -- drop the file in and use its prefix. An unknown tag
# raises with the available panels listed.
#   'gao'  -> ~2.4k brain-expressed genes (2031 present in the expression matrix)
#   'dear' -> ~8k  brain-expressed genes (7821 present)
# This changes the Gene PCs and everything downstream of them. It does NOT change the curated
# Gene_* maps in df_neuro (a fixed hypothesis-driven list) or any non-gene family.
gene_panel = 'dear'          # 'gao' or 'dear'
gene_tag   = f'_{gene_panel}'

notebook_data        = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                          output_type=output_type, use_resids=True,
                                          load_subject_data=True, n_regions=n_regions,
                                          gene_panel=gene_panel)
timescales_processor = notebook_data['timescales_processor']

nas_scratch_dir = figure_scratch_dir(dataset)


In [ ]:
# Cross-notebook GAM export written by timescales_analysis_fig3.ipynb (its GAM + bootstrap cell).
# Figure 4 fits nothing itself -- every downstream cell reads this.
task_scan  = 'tfMRICARITPA'

# These MUST match fig3's GAM config cell: together with task_scan they name the cache file, and
# fig3 writes one file per combination. gam_variant_tags is the SAME resolver fig3 uses, so the
# path below cannot drift from the one fig3 wrote; the values are cross-checked against those
# recorded inside the export as well, so a stale or renamed file fails loudly.
fit_recipe = 'gcv'  # 'historical' -> n_splines=4, lam=0.1 fixed; 'gcv' -> n_splines=5, GCV lam
absolute   = True   # deflection definition: True -> |task - rest|, False -> signed (task - rest)
demean     = True   # each series centred on its own mean before fitting

_tags        = gam_variant_tags(fit_recipe, absolute, demean)
n_splines    = _tags['n_splines']
recipe_tag   = _tags['recipe_tag']
absolute_tag = _tags['absolute_tag']
demean_tag   = _tags['demean_tag']

# Which INT signal this run of the figure characterises. fig3 exports all three, so switching here
# is enough -- the neurobiological coupling, stage maps, S-A statistics and PC characterisation
# below all follow signal_type, and every saved panel carries it in its filename.
signal_type = 'deflection'          # 'rest', 'task', or 'deflection'

# Which stage boundary splits stage 1 from stage 2:
#   'deflection' -> the shared deflection-derived boundary. Stage maps stay directly comparable
#                   across signals, and match the dashed boundary fig3 draws on every panel.
#   'signal'     -> the boundary detected on signal_type's own fit (a different age per signal).
STAGE_BOUNDARY = 'deflection'

SIGNAL_LABELS = signal_labels(absolute)   # both dicts are shared with fig3, so the two figures
SIGNAL_UNITS  = signal_units(absolute)    # cannot label the same signal differently
if signal_type not in SIGNAL_TYPES:
    raise ValueError(f'signal_type must be one of {list(SIGNAL_TYPES)}; got {signal_type!r}')
if STAGE_BOUNDARY not in ('deflection', 'signal'):
    raise ValueError(f"STAGE_BOUNDARY must be 'deflection' or 'signal'; got {STAGE_BOUNDARY!r}")
SIGNAL_LABEL = SIGNAL_LABELS[signal_type]
SIGNAL_UNIT  = SIGNAL_UNITS[signal_type]
# Same meaning as fig3's variant_tag (absolute x demean), so panel stems read identically across
# the two notebooks: ..._{task_scan}_{signal_type}{variant_tag}. Keeps the 12-cell
# absolute x demean x signal_type grid in distinct files instead of overwriting each other.
variant_tag  = _tags['variant_tag']

gam_file = gam_cache_path(os.path.join(timescales_processor.output_dir, 'results', dataset, 'gam'),
                          dataset, n_regions, task_scan, _tags)
if not os.path.exists(gam_file):
    raise FileNotFoundError(
        f'GAM export not found:\n  {gam_file}\n'
        'Run timescales_analysis_fig3.ipynb (through its GAM + bootstrap cell) first to generate it.')
_d = np.load(gam_file, allow_pickle=True).item()

# Exactly the keys Figure 4 consumes -- an export missing any of them predates this notebook.
_required = ['age_grid', 'sa_axis', 'df_behavior', 'n_splines', 'boot_seed', 'inflection_age',
             'rest_reg', f'{signal_type}_pred', f'{signal_type}_reg', f'{signal_type}_lams',
             f'{signal_type}_inflection_age']
_missing  = [k for k in _required if k not in _d]
if _missing:
    raise KeyError(f'GAM export {os.path.basename(gam_file)} is missing keys {_missing}; re-run '
                   "fig3's GAM + bootstrap cell with FORCE_RERUN = True to regenerate it.")

age_grid    = _d['age_grid']
sa_axis     = _d['sa_axis']           # full-parcellation axis (== fig3's sa_axis_reg when
                                      # unmasked, which the guard below requires)
df_behavior = _d['df_behavior']
rest_reg    = _d['rest_reg']          # (n_sub, n_regions) resting INT -> bootstrap nuisance term

# Per-signal inputs. sig_reg is what the bootstrap re-fits; sig_lams freezes the smoothing to the
# one fig3 selected for THIS signal, so resamples here reproduce fig3's fit rather than re-tuning.
sig_pred = _d[f'{signal_type}_pred']  # (n_regions, n_grid) fitted smooths
sig_reg  = _d[f'{signal_type}_reg']   # (n_sub, n_regions) subject data behind those smooths
sig_lams = _d[f'{signal_type}_lams']  # (n_regions,) frozen per-region smoothing
inflection_age = float(_d['inflection_age'] if STAGE_BOUNDARY == 'deflection'
                       else _d[f'{signal_type}_inflection_age'])

# The filename encodes most of these, but check the values recorded INSIDE the export too -- a
# hand-renamed or stale cache would otherwise be silently mislabelled in every panel below.
for _name, _want in [('absolute', bool(absolute)), ('demean', bool(demean)),
                     ('fit_recipe', fit_recipe), ('n_splines', n_splines),
                     ('task_scan', task_scan)]:
    if _name not in _d:
        continue                                     # older export; the filename still pins it
    _got = bool(_d[_name]) if isinstance(_want, bool) else _d[_name]
    if _got != _want:
        raise ValueError(f'{os.path.basename(gam_file)} was built with {_name}={_got!r}, but this '
                         f'cell asks for {_name}={_want!r}. Fix the flag or regenerate the cache.')

# fig4 works at the FULL parcellation: the surface panels render schaefer_{n_regions}-7 and every
# BrainSMASH null is built over all n_regions parcels. fig3's apply_tsnr_mask drops columns from
# the exported fits, which would misalign sig_pred against sa_axis (and against the surface) with
# no error until something broadcast-fails much further down. Refuse it up front.
if _d.get('region_mask_applied', False):
    raise ValueError(f'{os.path.basename(gam_file)} was built with fig3\'s apply_tsnr_mask on. '
                     'Re-run fig3 with apply_tsnr_mask = False.')
require_full_parcellation(sig_pred.shape[0], n_regions, os.path.basename(gam_file),
                          'Re-run fig3 with apply_tsnr_mask = False.')

print(f'Loaded GAM export <- {os.path.basename(gam_file)} '
      f'(absolute={absolute}, demean={demean})')
print(f'  signal_type={signal_type!r} ({SIGNAL_LABEL}); sig_pred {sig_pred.shape} '
      f'on age_grid {age_grid.shape}')
print(f'  stage boundary {inflection_age:.2f} yr (from {STAGE_BOUNDARY}) | this signal\'s own '
      f'{_d[f"{signal_type}_inflection_age"]:.2f} yr | deflection {_d["inflection_age"]:.2f} yr')
print(f'  sig_reg {sig_reg.shape} | rest_reg {rest_reg.shape} | df_behavior {df_behavior.shape}')


# Figure 4 — neurobiology and the developmental *rate* of task-evoked INT modulation

Two analyses share the GAM smooths loaded above, for whichever signal `signal_type` selects
(rest INTs, task INTs, or task-vs-rest deflections):

1. **Neurobiological coupling.** At each age, the regional pattern of the GAM's temporal derivative
   is correlated against a set of reference brain maps (genes, myelin), with significance
   from SA-preserving BrainSMASH surrogates of the reference map.
2. **S-A axis.** Per developmental stage, the annualized regional net change is correlated
   against the S-A axis — with a subject bootstrap and a null built from surrogates of the axis —
   and summarised by S-A decile.

The panels for both are produced by the three cells at the end and assembled externally.


In [ ]:
# ── Column-family inclusion toggles ──────────────────────────────────────────────────────────────
# Set a flag False to drop those maps (the family's DataFrame becomes None and the compute/plot
# cells skip it via their `is None` guards).
#   INCLUDE_GENE_MAPS      -> AHBA gene-expression maps (df_genes)        -> 'Genes' family
#   INCLUDE_OWN_MYELIN_MAP -> this dataset's own group-mean T1w/T2w map   -> 'Myelin' family
# WHICH maps within the Gene family appear is set by gene_curated below.
INCLUDE_GENE_MAPS      = True
INCLUDE_OWN_MYELIN_MAP = True

# ── Analysis hemisphere ─────────────────────────────────────────────────────────────────────────
# True -> every REFERENCE-MAP analysis is restricted to the LEFT hemisphere: the gene PCA,
# the gene / myelin coupling traces, the PC characterisation, and
# every BrainSMASH null behind them. AHBA sampling is overwhelmingly left-hemisphere, so this
# drops the sparsely-sampled right-hemi parcels instead of imputing them into the components.
#
# The S-A section is deliberately NOT restricted: the stage-1 / stage-2 INT maps, their surface
# panels, the S-A axis correlation and the S-A axis null all stay bilateral.
#
# A restricted run needs its own surrogates. The cache stem gains a '_L' suffix, so switching
# this on regenerates every reference-map null rather than silently serving the bilateral ones
# (the S-A axis null is shared, since that analysis is unchanged).
LEFT_HEMI_ONLY = False

# Components fitted for the Gene family. The PCA is run in this notebook (not in
# load_notebook_data), so this is the only place the component count is set.
N_PCS = 5

# The sign of a principal component is arbitrary: X ~ scores @ loadings.T, so negating BOTH leaves
# the fit, the variance explained and every |correlation| untouched -- only the labelling of which
# pole is "positive" changes. Flipping every component makes the more interpretable genes land on
# the positive pole across all PCs, which is easier to reason about. Applied inside fit_family_pca
# so scores, loadings, the catalogue maps, the developmental couplings, the reference-map
# correlations and the stability bootstrap all flip together -- never one without the others.
# NOTE: every reported correlation changes SIGN when this is toggled (magnitudes and p-values do
# not), so any written interpretation has to be re-read against the setting used.
PCA_FLIP_SIGN = False

# Subject bootstrap resamples, used by the S-A section. 500 = quick; raise to 5000+ for publication.
N_BOOT = 500

# BrainSMASH spatial surrogates -- the single knob for EVERY spatial null here (the neurobio
# reference maps and the S-A axis). Surrogates are cached per (map, n_regions, n_perms), so changing 
# this count regenerates them at the new count rather than reusing the old cache.
N_PERMS = 10000
# N_PERMS = 1000

# Shared null basis: the per-hemisphere geodesic distance matrix and the on-disk surrogate cache.
# Loaded once here because both compute cells below need them.
distance_matrix, hemi_labels, cache_dir = null_basis(timescales_processor, dataset, n_regions)
print(f'BrainSMASH basis: geodesic distances {distance_matrix.shape}, {N_PERMS} surrogates per map')
print(f'Surrogate cache: {cache_dir}')

# Parcel mask + filename suffix implementing LEFT_HEMI_ONLY. MASKING rather than subsetting is
# what the null machinery wants: generate_surrogates already works per hemisphere over the finite
# parcels only, so a map that is NaN on the right yields left-only surrogates built on the
# left-only geodesic block -- identical to subsetting the arrays, but every vector stays
# (n_regions,) so the bilateral S-A section and the surface plots need no changes at all.
# corr_with_null and deriv_map_spatial_null_p likewise reduce to the finite parcels themselves.
hemi_mask = (np.asarray(hemi_labels) == 'L') if LEFT_HEMI_ONLY else np.ones(n_regions, dtype=bool)
hemi_tag  = '_L' if LEFT_HEMI_ONLY else ''

def restrict_hemi(vals):
    """NaN out parcels outside the analysis hemisphere. A no-op when LEFT_HEMI_ONLY is False."""
    out = np.asarray(vals, dtype=float).copy()
    out[~hemi_mask] = np.nan
    return out

print(f'Analysis hemisphere: {"LEFT only" if LEFT_HEMI_ONLY else "bilateral"} '
      f'({int(hemi_mask.sum())}/{n_regions} parcels). Applies to the reference-map analyses '
      'only; the S-A stage section stays bilateral.')


## Molecular maps: genes

`gene_curated` is the only control over which maps enter the figure. The cell prints the full
catalogue of names it accepts.


## Panel + analysis support

Helpers for this notebook, gathered so the cells below stay readable. Collapse it.

In [ ]:
# ── Panel + analysis support ────────────────────────────────────────────────────────────────
# Every helper this notebook defines, collected in one place so the analysis below reads as a
# sequence of steps rather than a mix of steps and machinery. Collapse this cell when reading.
# These are notebook-specific by design: anything used by a second notebook belongs in src/.
# They resolve globals (config constants, loaded data) at CALL time, so definitions may refer
# to names bound further down the notebook.

# ── Helpers used by the cells below ──────────────────────────────────────────────────────────────
# save_panel now lives in src.plotting (shared with fig3); panels are written as PNG + SVG at
# 300 dpi with a tight bbox, the SVG staying text-editable thanks to svg.fonttype='none'.
def longest_sig_run(p, alpha):
    """Length (in age-grid points) of the longest run of consecutive p < alpha; 0 if none."""
    best = run = 0
    for hit in np.asarray(p) < alpha:
        run = run + 1 if hit else 0
        best = max(best, run)
    return best


def fit_family_pca(df, n_components, label, flip=False, row_mask=None):
    """Fit a family's PCA and keep everything needed to interpret and re-fit it.

    load_supporting_data returns the raw expression / receptor matrices only -- the components live
    here so that the fit, its interpretation and its bootstrap all sit in one place. Standardised
    and NaN-imputed, so scores come back for every parcel; parcels with no data at all are pure
    imputation and are masked out by the caller.

    Adds two things ``pca_with_nan_handling`` does not return: the feature names in loadings order
    (without which loadings cannot be mapped back to genes) and the input matrix (without which the
    stability bootstrap has nothing to re-fit).

    ``flip`` negates scores AND loadings together (see PCA_FLIP_SIGN). Both must move or the two
    stop describing the same component; ``input_data`` is deliberately left alone, since the data
    have no sign and the bootstrap re-fits from them and re-aligns to whichever reference it is
    given. Nothing else in the returned dict is sign-carrying for our purposes.

    ``row_mask`` (n_parcels,) restricts which parcels the component is FIT on -- this is how
    LEFT_HEMI_ONLY works. The fit must drop those rows rather than NaN them, because impute=True
    would otherwise fill them with column medians and let phantom parcels shape the components.
    Scores are scattered back to full parcel length with NaN outside the mask, so every consumer
    still sees an (n_regions,) map. ``input_data`` is the masked matrix (out-of-mask rows all-NaN,
    which is exactly what loading_pole_stability's all-NaN row drop expects, keeping the bootstrap
    on the same parcels as the fit); ``input_data_full`` keeps the unrestricted matrix for export.
    """
    X_full = np.asarray(df, dtype=float)
    mask   = np.ones(X_full.shape[0], dtype=bool) if row_mask is None else np.asarray(row_mask, dtype=bool)
    out = pca_with_nan_handling(X_full[mask], n_components=n_components, standardize=True,
                                impute=True)
    if flip:
        out['scores']   = -out['scores']
        out['loadings'] = -out['loadings']
    scores_full = np.full((X_full.shape[0], n_components), np.nan)
    scores_full[mask]     = out['scores']
    out['scores']         = scores_full
    out['row_mask']       = mask
    out['feature_names']  = list(df.columns)
    X_masked              = X_full.copy()
    X_masked[~mask]       = np.nan
    out['input_data']     = X_masked
    out['input_data_full'] = X_full
    print(f'{label} PCA: {X_full.shape[1]} features x {int(mask.sum())} parcels '
          f'({"LEFT hemisphere only" if not mask.all() else "bilateral"}) -> {n_components} '
          f'components ({", ".join(f"{v:.1%}" for v in out["explained_variance_ratio"])} variance)'
          + ('  [SIGN FLIPPED: every correlation below changes sign, magnitudes do not]'
             if flip else ''))
    return out


def loading_pole_stability(X, ref_loadings, n_boot, window, seed=0):
    """How reproducibly each feature re-enters each pole's top-``window`` under parcel resampling.

    The PCA is re-fit on bootstrap resamples of the parcels that carry data, and each re-fit's
    components are matched back to the reference fit by |correlation| of their loading vectors and
    then sign-aligned -- necessary because PCA is only defined up to sign, and components with
    similar explained variance can swap order between resamples.

    ``window`` is deliberately wider than the number of features plotted: it defines "belongs to this
    pole at all", so the eligible pool is larger than the panel and the top-N selection can be drawn
    from it. Stability is only comparable across runs that share the same ``window``.

    Returns (pos_freq, neg_freq), each (n_features, n_components) in [0, 1]: the fraction of re-fits
    in which the feature appeared in that component's ``window`` most positive / most negative
    loadings.
    """
    X    = np.asarray(X, dtype=float)
    rows = X[~np.isnan(X).all(axis=1)]                       # drop parcels with no data at all
    n_comp = ref_loadings.shape[1]
    pos, neg = (np.zeros(ref_loadings.shape) for _ in range(2))
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        boot = pca_with_nan_handling(rows[rng.choice(len(rows), len(rows), replace=True)],
                                     n_components=n_comp, standardize=True, impute=True)['loadings']
        for k in range(n_comp):
            corr = [np.corrcoef(ref_loadings[:, k], boot[:, m])[0, 1] for m in range(n_comp)]
            j    = int(np.argmax(np.abs(corr)))
            w    = boot[:, j] * np.sign(corr[j])
            pos[np.argsort(w)[::-1][:window], k] += 1
            neg[np.argsort(w)[:window], k] += 1
    return pos / n_boot, neg / n_boot


def catalog_add(catalog, name, values, *aliases):
    """Register one map under its canonical name plus any alternative spellings.

    ``name`` becomes the DataFrame column -- and therefore both the trace label and the BrainSMASH
    null-cache filename stem -- so it must stay stable. Aliases exist only so the curated lists can
    be written in whichever form reads best.
    """
    entry = (name, np.asarray(values, dtype=float))
    for key in (name, *aliases):
        catalog[key] = entry


def select_maps(catalog, curated, family):
    """The maps named in ``curated``, in that order, as a (n_regions x n_maps) DataFrame.

    Unknown names raise with the catalogue listed, so a typo fails loudly instead of quietly
    shrinking the panel. A map named twice (directly and via an alias) is kept once.
    """
    unknown = [k for k in curated if k not in catalog]
    if unknown:
        raise KeyError(f'{family}: unknown map name(s) {unknown}.\nAvailable names:\n  '
                       + '\n  '.join(sorted({nm for nm, _ in catalog.values()})))
    out = {}
    for key in curated:
        name, values = catalog[key]
        out.setdefault(name, values)
    return pd.DataFrame(out)


# Gene maps only exist at Schaefer-400; at any other resolution that family is skipped
# and only the myelin family runs. Downstream cells guard on `df_* is None`.
df_neuro_full = notebook_data['supporting_data']['df_neuro'].copy()


In [ ]:
# ── Curated gene panels (AHBA symbols; all present in df_neuro as Gene_*) ────────────────────────
# Hypothesis-driven sets, kept as named vocabulary for gene_curated below.
genes_original = ['GRIN2B', 'GRIN2A', 'GABRA3', 'GABRA1', 'PVALB', 'PRR5', 'KCNA3', 'SCN1A']

genes_recurrent_excitation = [        # recurrent / glutamatergic synaptic strength
      'GRIN1', 'GRIN2A', 'GRIN2B',            # NMDA receptor subunits
      'GRIA1', 'GRIA2', 'GRIA3', 'GRIA4',     # AMPA receptor subunits
      'GRM5',                                 # metabotropic glutamate receptor 5
      'CAMK2A', 'CAMK2B',                     # postsynaptic Ca2+/calmodulin kinase
      'DLG4',                                 # PSD-95
      'SYNGAP1', 'SHANK3', 'HOMER1',          # postsynaptic scaffolding / signaling
]

genes_presynaptic_exocytosis = [      # presynaptic / calcium-regulated exocytosis
      'SYT1',                                 # synaptotagmin-1 (Ca2+ sensor)
      'STX1A', 'STXBP1', 'SNAP25', 'VAMP2',   # SNARE complex
      'RIMS1',                                # active-zone release machinery
      'CACNA1A', 'CACNA1B',                   # presynaptic Ca2+ channels (P/Q, N-type)
      'SV2A',                                 # synaptic vesicle glycoprotein 2A
      'SYN1', 'SYP',                          # synapsin-1, synaptophysin
      'CPLX1', 'CPLX2',                       # complexins
]

# ── Which gene maps go into the figure ──
# One list, in plotting order, mixing freely:
#   AHBA gene symbols  'GRIN2B'        (also accepted: the raw column 'Gene_GRIN2B')
#   expression PCs     'PCA_Gene-PC1'  (also accepted: the short form 'Gene-PC1')
# gene_curated = ['PCA_Gene-PC1', 'PCA_Gene-PC2', 'PCA_Gene-PC3', 'PCA_Gene-PC4', 'PCA_Gene-PC5']
gene_curated = ['PCA_Gene-PC1', 'PCA_Gene-PC2', 'PCA_Gene-PC3']
# gene_curated = genes_original
# gene_curated = list(dict.fromkeys(
#       genes_original + genes_recurrent_excitation + genes_presynaptic_exocytosis))

if n_regions != 400 or not INCLUDE_GENE_MAPS:
    df_genes = None
    print(f'n_regions={n_regions} != 400; skipping the gene family (Schaefer-400 only).'
          if n_regions != 400 else 'INCLUDE_GENE_MAPS=False; excluding the gene family.')
else:
    # AHBA coverage: the parcels where the gene panel has data (the rest, mostly right hemisphere,
    # are NaN and dropped per-map downstream). Taken from the mask load_supporting_data computes, so
    # it does not depend on a gene being in gene_curated -- a PC-only selection masks identically.
    _gene_cov  = np.asarray(notebook_data['supporting_data']['gene_expression_nanmask'], dtype=bool)
    _gene_cols = [c for c in df_neuro_full.columns if c.startswith('Gene_')]

    # ── Catalogue: every gene map that gene_curated may name ──
    gene_catalog = {}
    for _col in _gene_cols:
        catalog_add(gene_catalog, _col.replace('Gene_', ''),
                    df_neuro_full[_col].astype(float).values, _col)
    # Gene-expression components, fit here on the full Gao brain-gene panel (~2k genes), not on the
    # curated panel above. The fit imputes parcels with no AHBA data, so the scores are masked back
    # to the AHBA coverage: the PCs are then tested on the same regions as the individual genes.
    gene_pca = fit_family_pca(notebook_data['supporting_data']['gene_expression_brain'],
                              N_PCS, 'Gene', flip=PCA_FLIP_SIGN, row_mask=hemi_mask)
    for _i in range(N_PCS):
        _vals = gene_pca['scores'][:, _i].astype(float).copy()
        _vals[~_gene_cov] = np.nan
        catalog_add(gene_catalog, f'Gene-PC{_i + 1}', _vals, f'PCA_Gene-PC{_i + 1}')

    gene_catalog_genes = [c.replace('Gene_', '') for c in _gene_cols]
    gene_catalog_pcs   = [f'Gene-PC{_i + 1}' for _i in range(N_PCS)]
    _eff_cov = int((_gene_cov & hemi_mask).sum())     # parcels actually entering the analysis
    print(f'Gene catalogue over {n_regions} parcels: {len(gene_catalog_genes)} AHBA genes, '
          f'{len(gene_catalog_pcs)} PCs; AHBA coverage {int(_gene_cov.sum())} parcels '
          f'({n_regions - int(_gene_cov.sum())} NaN, dropped per-map downstream)'
          + (f'; {_eff_cov} after the LEFT-hemisphere restriction.' if LEFT_HEMI_ONLY else '.'))
    print('  genes:', ', '.join(gene_catalog_genes))
    print('  PCs  :', ', '.join(gene_catalog_pcs) or '(none)')

    df_genes = select_maps(gene_catalog, gene_curated, 'Genes')
    print(f'Gene selection ({df_genes.shape[1]}): {", ".join(df_genes.columns)}')

# ── Myelin family: this dataset's own group-mean T1w/T2w map ─────────────────────────────────────
# The parcel-wise mean of the per-subject myelin stack, added to df_neuro as '<DATASET>_Myelin'
# by load_notebook_data (HCP-YA / HCP-D with load_subject_data=True only). The column name is a
# contract with the BrainSMASH null cache, so it must stay as-is.
df_myelin = pd.DataFrame(index=np.arange(n_regions))

own_myelin_col = f"{dataset.replace('-', '')}_Myelin"          # 'HCP-D' -> 'HCPD_Myelin'
if not INCLUDE_OWN_MYELIN_MAP:
    print(f'INCLUDE_OWN_MYELIN_MAP=False; excluding {own_myelin_col} from the myelin family.')
elif own_myelin_col in df_neuro_full.columns:
    _own = df_neuro_full[own_myelin_col].astype(float).values
    df_myelin[own_myelin_col] = _own
    print(f'Added {own_myelin_col} (group-mean T1w/T2w myelin); '
          f'coverage {int(np.isfinite(_own).sum())}/{n_regions} parcels.')
else:
    print(f'{own_myelin_col} not in df_neuro; skipping the own-myelin map.')

if df_myelin.shape[1] == 0:
    df_myelin = None
    print('No myelin map selected/available; excluding the myelin family.')


# Neurobiological coupling to the developmental rate — compute

In [ ]:
# ── corr(brain map × kth GAM derivative)(age), with a pointwise SA-preserving-null p ─────────────
# Per map and per derivative order: the point-estimate r(age) and a two-sided spatial-null p(age)
# from BrainSMASH surrogates OF THE REFERENCE MAP. Significance here is spatial specificity, and it
# is pointwise -- uncorrected across age and across maps.
DERIV_ORDERS = (1, 2)

# ── Column families (each a DataFrame of length-n_regions maps) ──
neuro_families = {}
if df_genes is not None:
    neuro_families['Genes'] = df_genes
if df_myelin is not None:
    neuro_families['Myelin'] = df_myelin

# key 'Family::Map' -> (n_regions,) values. The map name doubles as the BrainSMASH cache stem, so
# pre-computed nulls in results/<dataset>/brainsmash_nulls/ are reused (GRIN2B, Gene-PC1,
# HCPD_Myelin, ...). Maps without one are generated and cached here.
# restrict_hemi is applied HERE, once. This dict is the single point where reference maps enter
# the coupling analysis, so genes and myelin are treated identically
# (a no-op when LEFT_HEMI_ONLY is False). Anything added as a new consumer of the family
# DataFrames must apply restrict_hemi itself -- the DataFrames are left unrestricted.
_maps = {f'{fam}::{col}': restrict_hemi(df[col].astype(float).values)
         for fam, df in neuro_families.items() for col in df.columns}
# Cache-stem tags. hemi_tag keeps restricted surrogates in their own files: the null for a
# left-only map is built over left parcels only and must never be served from the bilateral
# cache. gene_tag is applied ONLY to the gene PCs -- they are the sole maps whose VALUES depend
# on the brain-gene panel. An individual gene map (GRIN2B, ...) is the same map under 'gao' and
# 'dear', so tagging it would regenerate an identical null under a second name.
_panel_dep = set(gene_catalog_pcs) if df_genes is not None else set()
_stem = {k: (k.split('::')[1].replace(' ', '_')
             + (gene_tag if k.split('::')[1] in _panel_dep else '')
             + hemi_tag)
         for k in _maps}
print(f'Families: {list(neuro_families)}; {len(_maps)} maps; orders {DERIV_ORDERS}; '
      f'{N_PERMS} SA-preserving surrogates each '
      f'({"LEFT hemisphere only" if LEFT_HEMI_ONLY else "bilateral"}); '
      f'brain-gene panel {gene_panel!r}.')

_miss = [k for k in _maps if not os.path.exists(os.path.join(
    cache_dir, f'{_stem[k]}_kind-geodesic_regions-{n_regions}_perms-{N_PERMS}_nulls.npy'))]
print(f'Surrogate cache: {len(_maps) - len(_miss)}/{len(_maps)} pre-computed; '
      f'{len(_miss)} to generate -> {[_stem[k] for k in _miss]}')
_surr = {k: generate_surrogates(v, name=_stem[k], n_regions=n_regions, n_perms=N_PERMS,
                                distance_matrix=distance_matrix, hemi=hemi_labels,
                                cache_dir=cache_dir, seed=0)
         for k, v in _maps.items()}

# ── r(age) + pointwise spatial-null p(age) for every order × map ──
r_point, r_pval = {}, {}
for order in DERIV_ORDERS:
    _deriv = grid_derivatives(sig_pred, age_grid, order)
    for k, vals in _maps.items():
        r_point[f'd{order}|{k}'], r_pval[f'd{order}|{k}'] = deriv_map_spatial_null_p(
            _deriv, vals, _surr[k])
_n_sig = int(sum(np.nansum(v < 0.05) for v in r_pval.values()))
print(f'Done: {len(r_pval)} (order × map) traces; {_n_sig} age-points at p<0.05 across all traces.')


# S-A axis — annualized per-stage net change — compute

Per region, the signed net change in the selected signal (`signal_type`) within each stage
(stage 1 = age ≤ inflection, stage 2 = age ≥ inflection), **annualized** to Δ per year so the
two stages — which span different numbers of years — are comparable. Each stage map is correlated with the S-A axis (subject-bootstrap CI +
BrainSMASH null) and summarised by S-A decile with a bootstrap test against 0.


In [ ]:
# ── Annualized per-stage net change vs the S-A axis (regional + per-decile) ──────────────────────
# Metric (src.gams.regional_stage_net_change, annualized=True): per region, the integrated rate over
# the stage divided by the stage duration -> mean Δ per year in signal_type's units. Both stages
# share ONE bootstrap of the regional GAMs, since bootstrap_metric_sa_corr evaluates every
# requested metric per resample.
#
# The spatial null surrogates the S-A AXIS, not the stage map. The stage map is the empirical
# quantity under test; the reference map is what gets re-randomised with matched spatial
# autocorrelation, asking "is this coupling stronger than the axis's own smoothness would produce".
# One surrogate set of the axis therefore serves both stages, and never depends on the GAM fit.
SA_METRICS     = {1: 'early_net_annual', 2: 'late_net_annual'}   # stage -> src.gams metric name
SA_CORR_METHOD = 'pearson'                                       # Pearson throughout for
                                                                 # brain-map coupling, matching
                                                                 # fig3's SA_CORR_METHOD
SA_DECILES     = 10
SA_SYM         = 'rho' if SA_CORR_METHOD == 'spearman' else 'r'
_corr_fn       = sp.stats.spearmanr if SA_CORR_METHOD == 'spearman' else sp.stats.pearsonr

# Subject bootstrap: one pass, all stage metrics. lam / n_splines / seed are frozen to the fig3 fit.
# build_nuisance is the same call fig3 makes, so these refits reproduce fig3's model rather than
# a lookalike: [sex, head motion, each subject's mean resting INT].
_nuis = build_nuisance(df_behavior, rest_reg, dataset)
_sa_boot = bootstrap_metric_sa_corr(
    sig_reg, df_behavior['age'].values, age_grid, sa_axis, metrics=tuple(SA_METRICS.values()),
    method=SA_CORR_METHOD, inflection=inflection_age, n_splines=n_splines,
    lam=sig_lams, nuisance=_nuis, n_boot=N_BOOT, random_state=_d['boot_seed'],
    n_jobs=-1, verbose=True)

sa_axis_surr = generate_surrogates(sa_axis, name='sa_axis', n_regions=n_regions, n_perms=N_PERMS,
                                   distance_matrix=distance_matrix, hemi=hemi_labels,
                                   cache_dir=cache_dir, seed=0)

# sa_res[stage] = point-estimate map, r vs S-A + bootstrap CI, BrainSMASH null, decile summary.
sa_res = {}
for stage, metric in SA_METRICS.items():
    _map = sa_metric_map(metric, sig_pred, age_grid, inflection=inflection_age)
    _f   = np.isfinite(_map) & np.isfinite(sa_axis)
    sa_res[stage] = {
        'metric': metric,
        'label':  f'Annualized stage-{stage} net change (Δ{SIGNAL_UNIT} / yr)',
        'map':    _map,
        'r':      _corr_fn(_map[_f], sa_axis[_f]).statistic,
        'ci':     tuple(np.nanpercentile(_sa_boot[f'r|{metric}'], [2.5, 97.5])),
        'null':   corr_with_null(_map, sa_axis, sa_axis_surr, method=SA_CORR_METHOD),
        # Decile means reuse the bootstrap map distribution (subject-resampled GAM refits); this is
        # a test of "is the mean change in this decile != 0", NOT a spatial-enrichment null.
        'dec':    sa_decile_change(_map, _sa_boot[f'map|{metric}'], sa_axis, n_deciles=SA_DECILES),
    }

for stage, res in sorted(sa_res.items()):
    dec = res['dec']
    print(f"{res['label']} vs S-A axis ({SA_CORR_METHOD}); n_boot={N_BOOT}, "
          f'{N_PERMS} BrainSMASH surrogates of the S-A axis:')
    print(f"  regional: {SA_SYM}={res['r']:+.3f}  boot95%=[{res['ci'][0]:+.3f},{res['ci'][1]:+.3f}]  "
          f"p_smash={res['null']['p_smash']:.4f}")
    print(f'  per-decile mean (Δ/yr), bootstrap CI + p vs 0  (1=sensorimotor -> {SA_DECILES}=association):')
    for i in range(SA_DECILES):
        print(f"    decile {dec['decile'][i]:2d} (n={dec['n'][i]:3d}): {dec['mean'][i]:+.4f} "
              f"[{dec['ci_lo'][i]:+.4f}, {dec['ci_hi'][i]:+.4f}]  p={dec['p_boot'][i]:.3f}"
              + (' *' if dec['sig'][i] else ''))


# Figure 4 — panel sets (assembled externally)

Three cells, each producing one set of panels at its own size, saved as **PNG + SVG**
(`svg.fonttype='none'` keeps the text editable) for assembly in Illustrator:

1. **Stage maps** — one surface figure per stage.
2. **S-A statistics** — one 2×1 figure per stage: regional scatter over the decile bootstrap test.
3. **Coupling + PC composition** — one figure combining what were separate panel sets. Left: the
   coupling traces across age (with a marker at each map's peak) over the regional scatter behind
   each marker. Right: one loading-heatmap column per PC. It lives further down, after the PC
   characterisation compute cell whose outputs it needs.

Each cell is independently re-sizable; sizing knobs are the constants at the top of each.


In [ ]:
# ── Panel set 1: annualized per-stage net-change surface maps (one figure per stage) ─────────────
# SURF_SHARE_SCALES=True puts both stages on one symmetric colour scale, so intensity is comparable
# when the panels sit side by side; False lets each stage use its own 95th-percentile limit (note
# the stages then look equally strong even where their magnitudes differ).
SURF_FIGSIZE      = (2.25, 2.5)     # per-stage figure size (inches)
SURF_SHARE_SCALES = False

_maps_all = np.concatenate([sa_res[s]['map'] for s in sa_res])

for stage, res in sorted(sa_res.items()):
    vmax = np.round(np.nanpercentile(np.abs(_maps_all if SURF_SHARE_SCALES else res['map']), 95), 3)

    fig = plt.figure(figsize=SURF_FIGSIZE, dpi=300)
    plot_brain_surface_data_single(res['map'], fig, gridspec.GridSpec(1, 1, figure=fig)[0, 0],
                                   parcellation=f'schaefer_{n_regions}-7', cmap='RdBu_r',
                                   vmin=-vmax, vmax=vmax, symmetric_cbar=True,
                                   cbar_label=f'Δ{SIGNAL_UNIT} / yr', fontsize=8)
    _path = save_panel(fig, nas_scratch_dir, f'fig4_stage{stage}_map_{task_scan}_{signal_type}{variant_tag}')
    print(f'stage {stage}: vmax={vmax:.4g} -> {os.path.basename(_path)}.{{png,svg}}')
    plt.show()


In [ ]:
# ── Panel set 2: S-A statistics per stage (2×1: regional scatter over decile bootstrap) ──────────
# Top: every region's annualized stage net change vs the S-A axis. The stats box shows the
# correlation and the SA-axis-surrogate p (NOT the parametric p) -- note a permutation p can only
# resolve to 1/N_PERMS, so it is floored rather than printed as zero.
# Bottom: the same map by S-A decile -- bar = decile mean, whisker = subject-bootstrap 95% CI,
# filled + starred = CI excludes 0.
SA_FIGSIZE       = (1.25, 2.75)    # per-stage figure size (inches)
SA_SHARE_SCALES  = False           # common scatter / decile y-ranges across stages

_sa_cmap = register_custom_colormaps()['snap_sydnor_sa']
_all_map = np.concatenate([sa_res[s]['map'] for s in sa_res])
_sc_lo, _sc_hi = np.nanmin(_all_map), np.nanmax(_all_map)
_dec_lo = min(np.nanmin(sa_res[s]['dec']['ci_lo']) for s in sa_res)
_dec_hi = max(np.nanmax(sa_res[s]['dec']['ci_hi']) for s in sa_res)

for stage, res in sorted(sa_res.items()):
    dec = res['dec']
    fig, (ax, axd) = plt.subplots(2, 1, figsize=SA_FIGSIZE, dpi=300,
                                  gridspec_kw={'height_ratios': [1.0, 0.85], 'hspace': 0.55})

    # top: regional scatter vs the S-A axis
    fin = np.isfinite(res['map']) & np.isfinite(sa_axis)
    plot_correlation(pd.Series(sa_axis[fin]), pd.Series(res['map'][fin]), ax, x_label='S-A axis',
                     y_label='Δ/yr', method=SA_CORR_METHOD, show_stats=True, fontsize=8,
                     custom_pvalue=max(res['null']['p_smash'], 1 / N_PERMS))
    if SA_SHARE_SCALES:
        ax.set_ylim(_sc_lo - 0.06 * (_sc_hi - _sc_lo), _sc_hi + 0.06 * (_sc_hi - _sc_lo))
    ax.tick_params(labelsize=7, pad=2)

    # bottom: decile means with subject-bootstrap CI
    x, colors = dec['decile'], _sa_cmap(np.linspace(0, 1, len(dec['decile'])))
    for i, xi in enumerate(x):
        axd.bar(xi, dec['mean'][i], width=0.8, color=colors[i], edgecolor='k', linewidth=0.5,
                alpha=1.0 if dec['sig'][i] else 0.30, zorder=2)
    axd.errorbar(x, dec['mean'],
                 yerr=np.vstack([dec['mean'] - dec['ci_lo'], dec['ci_hi'] - dec['mean']]),
                 fmt='none', ecolor='k', elinewidth=0.8, capsize=2, zorder=3)
    _pad = 0.04 * (np.nanmax(dec['ci_hi']) - np.nanmin(dec['ci_lo']))
    for i, xi in enumerate(x):
        if dec['sig'][i]:
            up = dec['mean'][i] > 0
            axd.text(xi, dec['ci_hi'][i] + _pad if up else dec['ci_lo'][i] - _pad, '*',
                     ha='center', va='bottom' if up else 'top', fontsize=8)
    axd.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
    axd.set_xticks(x)
    axd.set_xlabel('S-A decile', fontsize=8)
    axd.set_ylabel('Δ/yr', fontsize=8)
    axd.tick_params(labelsize=7, pad=2)
    if SA_SHARE_SCALES:
        axd.set_ylim(_dec_lo - 0.22 * (_dec_hi - _dec_lo), _dec_hi + 0.22 * (_dec_hi - _dec_lo))
    else:
        axd.margins(y=0.18)

    _path = save_panel(fig, nas_scratch_dir, f'fig4_stage{stage}_sa_{task_scan}_{signal_type}{variant_tag}')
    print(f"stage {stage}: {SA_SYM}={res['r']:+.3f}, p_smash={res['null']['p_smash']:.4f} "
          f'-> {os.path.basename(_path)}.{{png,svg}}')
    plt.show()


# PC characterisation — pole genes and coupling to a reference map

For the gene family:

* **both poles of each PC** — the top-loading genes at the positive AND negative end. A component is
  a contrast, so sorting by |loading| hides half of it; the interpretable biology is often at the
  end that does not win on magnitude.
* **bootstrap stability** — the PCA is re-fit over resampled parcels, and a gene is only eligible
  for a pole if it re-enters that pole often enough. Unstable genes are dropped, not shaded.
* **corr(PC score map, `PC_REF_MAP`)** — how each PC lines up with the group-mean myelin map.

Two cells: the first runs the bootstrap and the reference-map nulls, the second draws the
**combined figure** — developmental coupling on the left, these PC loadings on the right.
Restyling therefore costs nothing; only re-run the first cell if one of its own knobs changes.


In [ ]:
# ── PC characterisation, part 1: bootstrap + reference-map correlations (the slow part) ─────────
# Split from the plotting cell below so re-styling the figure never re-runs the parcel bootstrap or
# the surrogate correlations. Re-run this cell only when one of ITS knobs changes; everything about
# how the figure looks lives in the next cell.
PC_REF_MAPS         = ['HCPD_Myelin', 'SA_Axis']   # df_neuro columns each PC is correlated against
                                     # (tabulated, not plotted). 'SA_Axis' is stored as ranks
                                     # 1..n_regions, so only the OTHER side of each correlation
                                     # is affected by the pearson/spearman choice.
PC_CORR_METHOD      = 'pearson'      # Pearson throughout for brain-map coupling
PC_N_BOOT           = 500            # parcel-bootstrap PCA re-fits behind the stability estimate
PC_STABILITY_WINDOW = 40             # the rank window stability is measured against. Must exceed
                                     # the plotting cell's PC_TOP_N so the stable pool can fill
                                     # every slot -- keeping it an absolute number rather than a
                                     # multiple of PC_TOP_N is what lets PC_TOP_N change without
                                     # re-running this cell. Set it much wider than PC_TOP_N (>=3x)
                                     # and every top-N gene passes, i.e. the filter goes inert.
PC_MIN_STABILITY    = 0.5            # a gene is eligible if it re-enters this pole's top
                                     # PC_STABILITY_WINDOW in at least this fraction of re-fits.
                                     # 0.0 disables the filter (plain top-N by loading).

_family_df = df_genes
if _family_df is None:
    raise RuntimeError('The gene family was not built (INCLUDE_GENE_MAPS is off, or '
                       'n_regions != 400), so there is nothing to characterise.')
_catalog = gene_catalog
_pca     = gene_pca      # fitted in the molecular cell above
_missing_ref = [m for m in PC_REF_MAPS if m not in df_neuro_full.columns]
if _missing_ref:
    raise KeyError(f'PC_REF_MAPS entries not found in df_neuro: {_missing_ref}')

# PCs to characterise: those actually selected into the figure, else every stored component.
pc_names = [c for c in _family_df.columns if re.fullmatch(r'.+-PC\d+', c)]
if not pc_names:
    pc_names = [f'Gene-PC{i + 1}' for i in range(N_PCS)]
    print('No gene PCs in the current selection; characterising all fitted PCs instead.')
pc_idx      = [int(c.rsplit('PC', 1)[1]) - 1 for c in pc_names]  # 'Gene-PC2' -> loadings column 1
pc_loadings = _pca['loadings']
pc_features = np.asarray(_pca['feature_names'])
pc_evr      = _pca['explained_variance_ratio']
print(f'Genes: characterising {len(pc_names)} PCs ({", ".join(pc_names)}) over '
      f'{len(pc_features)} features; PCA explains '
      f'{", ".join(f"{pc_evr[i]:.1%}" for i in pc_idx)} of variance.')

# ── Loading-pole stability: how reproducibly each gene re-enters each pole under parcel resampling ──
print(f'Bootstrapping the gene PCA ({PC_N_BOOT} parcel resamples, stability measured against '
      f"each pole's top {PC_STABILITY_WINDOW}) ...")
pc_pos_freq, pc_neg_freq = loading_pole_stability(_pca['input_data'], pc_loadings, PC_N_BOOT,
                                                  PC_STABILITY_WINDOW, seed=0)

# ── corr(PC score map, each reference map), null from surrogates of that reference ──
# p_smash is floored at 1/N_PERMS: a permutation test cannot resolve below its own resolution, so a
# raw 0 is reported as "< 1/N_PERMS" rather than as zero.
_pc_corr = {}
for _ref_name in PC_REF_MAPS:
    _ref      = restrict_hemi(df_neuro_full[_ref_name].astype(float).values)
    _ref_surr = generate_surrogates(_ref, name=f'{_ref_name}{hemi_tag}',
                                    n_regions=n_regions, n_perms=N_PERMS,
                                    distance_matrix=distance_matrix, hemi=hemi_labels,
                                    cache_dir=cache_dir, seed=0)
    for pc in pc_names:
        _pc_corr[(pc, _ref_name)] = corr_with_null(_catalog[pc][1], _ref, _ref_surr,
                                                   method=PC_CORR_METHOD)

pc_ref_table = pd.DataFrame(index=pc_names)
pc_ref_table['var_explained'] = [pc_evr[i] for i in pc_idx]
pc_ref_table['n_parcels']     = [_pc_corr[(pc, PC_REF_MAPS[0])]['n'] for pc in pc_names]
for _m in PC_REF_MAPS:
    pc_ref_table[f'{_m}_r'] = [_pc_corr[(pc, _m)]['r'] for pc in pc_names]
    pc_ref_table[f'{_m}_p'] = [max(_pc_corr[(pc, _m)]['p_smash'], 1 / N_PERMS) for pc in pc_names]
print('Done. Stability frequencies + reference-map correlations are ready for the plotting cell.')


In [ ]:
# ── Panel sets 3 + 4 combined: PC loadings (left) + developmental coupling (right) ───────────────
# One figure from two nested grids that share a height.
#   LEFT (1 row x n_PCs): the PC_TOP_N genes at each pole of every PC, most negative at the bottom
#        and most positive at the top. Cell colour = loading; gene name and value printed inside.
#   RIGHT (2 rows x n_maps): row 1 = corr(map x DERIV_ORDER_SHOW-th derivative of the signal_type
#        GAM) against age, one trace per map in PANEL_FAMILY, faded across the full age range and
#        bold where the pointwise SA-preserving-null p < ALPHA_SIG, with a filled marker at the age
#        where |r| peaks. Row 2 = the scatter behind each marker, one column per trace in the
#        matching colour.
# Read across: a loading column on the left and a trace on the right are the same component. When
# both sides show the same components the heatmap titles are drawn in the trace colour to make that
# link explicit. Left says WHAT a component is made of, right says WHEN it couples to development.
#
# Everything is read from the two compute cells above; nothing is re-fit, so this cell is cheap to
# re-run while tuning the layout.

# ---- coupling block (right) ----
PANEL_FAMILY     = 'Genes'      # ONE family per figure; the others are computed but not plotted
PANEL_WIDTH      = 1.1          # width of one coupling column (inches)
ROW_HEIGHTS      = (2.5, 1.5)  # (trace row, scatter row) heights in inches; sets the figure
                                # height. Same 1.6:1 ratio as before, scaled up so the right
                                # block has room for 2 x PC_TOP_N rows at PC_CELL_FONTSIZE.
DERIV_ORDER_SHOW = 1            # 1 = rate of change, 2 = acceleration
ALPHA_SIG        = 0.05
SIG_ONLY         = False
MIN_SIG_RUN      = 1
PANEL_PALETTE    = 'snap_trio'  # 'snap_trio' = the lab's three categorical colours (raspberry
                                # blush / starry-night blue / north-sea green), for <= 3 maps;
                                # 'viridis' = the sequential ramp, for any number of maps.
ZOOM_SHARE_Y     = True         # row 2 shares one y-axis: tick labels on the leftmost panel only,
                                # which frees the width the duplicated labels were eating and makes
                                # the slopes directly comparable. False = per-panel y-axes.
GS_SPACING       = dict(hspace=0.55, wspace=0.25)   # within the coupling block
TICK_MAX         = dict(x=3, y=4)                   # max ticks per row-2 axis (narrow panels)

# ---- PC loading block (left) ----
PC_TOP_N         = 20                # genes shown per pole (so 2 x PC_TOP_N cells per PC)
PC_PANEL_WIDTH   = 1.2              # width of one PC column (inches); height is inherited from
                                     # the coupling block, so the two line up. Wide enough for
                                     # the longest gene symbol plus its value at PC_CELL_FONTSIZE.
PC_HEATMAP_CMAP  = 'snap_blue_pink'  # diverging map, centred on zero
PC_CELL_FONTSIZE = 6                 # in-cell gene name / loading text
PC_ALPHA_SIG     = 0.05              # * threshold in the reference-map table

# ---- how the two blocks sit together ----
# The coupling block now sits on the RIGHT, so its y-axis labels ('Pearson r ...', '1st
# derivative') are interior and the block gap has to clear them. The PC block on the left has no
# tick labels at all, so the figure's left margin can be tight.
BLOCK_WSPACE     = 0.34         # gap between the two blocks (fraction of a block width)
FIG_MARGINS      = dict(left=0.02, right=0.99, top=0.90, bottom=0.13)

if DERIV_ORDER_SHOW not in DERIV_ORDERS:
    raise ValueError(f'DERIV_ORDER_SHOW={DERIV_ORDER_SHOW} is not in DERIV_ORDERS={DERIV_ORDERS}; '
                     'add it there and re-run the neurobiology compute cell.')
if PANEL_FAMILY not in neuro_families:
    raise KeyError(f'PANEL_FAMILY {PANEL_FAMILY!r} is not a computed family: {list(neuro_families)}')
if PC_TOP_N >= PC_STABILITY_WINDOW:
    raise ValueError(f'PC_TOP_N ({PC_TOP_N}) must be below PC_STABILITY_WINDOW '
                     f'({PC_STABILITY_WINDOW}), otherwise the stable pool cannot fill the panel. '
                     'Raise the window in the PC compute cell and re-run it.')
if set(neuro_families) - {PANEL_FAMILY}:
    print(f'Computed but not plotted: {sorted(set(neuro_families) - {PANEL_FAMILY})}')

_key   = lambda col: f'd{DERIV_ORDER_SHOW}|{PANEL_FAMILY}::{col}'
_cols  = list(neuro_families[PANEL_FAMILY].columns)
if SIG_ONLY:
    _keep = [c for c in _cols if longest_sig_run(r_pval[_key(c)], ALPHA_SIG) >= MIN_SIG_RUN]
    _drop = [c for c in _cols if c not in _keep]
    print(f'{PANEL_FAMILY}: keeping {len(_keep)}/{len(_cols)} maps with a significant section '
          f'(>= {MIN_SIG_RUN} consecutive points)' + (f'; dropped {_drop}' if _drop else ''))
    _cols = _keep
if not _cols:
    raise ValueError(f'No maps left to plot for {PANEL_FAMILY} (SIG_ONLY dropped them all).')

# One colour serves both rows of the coupling block, so a row-1 trace and its row-2 scatter always
# match. Note the assignment is positional: colours follow the maps actually drawn, so dropping a
# trace via SIG_ONLY / MIN_SIG_RUN re-assigns them.
_snap_trio = get_snap_colors(cat_trio=True, as_list=True)      # categorical, distinguishable
if PANEL_PALETTE == 'snap_trio' and len(_cols) <= len(_snap_trio):
    _colors = _snap_trio[:len(_cols)]
else:
    if PANEL_PALETTE == 'snap_trio':
        print(f'PANEL_PALETTE={PANEL_PALETTE!r} carries {len(_snap_trio)} colours but {len(_cols)} '
              'maps are plotted; falling back to viridis.')
    _colors = plt.cm.viridis(np.linspace(0.0, 0.9, len(_cols)))
_ord_label = {1: '1st', 2: '2nd'}
_deriv     = grid_derivatives(sig_pred, age_grid, DERIV_ORDER_SHOW)      # (n_regions, n_grid)
_peak      = {c: int(np.nanargmax(np.abs(r_point[_key(c)]))) for c in _cols}   # age index per map

# The colour link is only honest when both sides are showing the same components in the same order.
_linked = _cols == pc_names
if not _linked:
    print(f'NOTE: the coupling block shows {_cols} but the PC block shows {pc_names}; titles are '
          'left black rather than implying a colour correspondence that does not hold.')

# ── Figure: [ left: one loading column per PC ] [ right: traces over scatters ] ──
_pc_w       = PC_PANEL_WIDTH * len(pc_names)
_coupling_w = PANEL_WIDTH * len(_cols)
fig = plt.figure(figsize=(_pc_w + _coupling_w, sum(ROW_HEIGHTS)), dpi=300)
gs_outer    = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[_pc_w, _coupling_w],
                                wspace=BLOCK_WSPACE, **FIG_MARGINS)
gs_pc       = gridspec.GridSpecFromSubplotSpec(1, len(pc_names), subplot_spec=gs_outer[0],
                                               wspace=0.12)
gs_coupling = gridspec.GridSpecFromSubplotSpec(2, len(_cols), subplot_spec=gs_outer[1],
                                               height_ratios=list(ROW_HEIGHTS), **GS_SPACING)

# ── Right row 1: coupling traces, with a marker at the age each row-2 panel zooms into ──
ax = fig.add_subplot(gs_coupling[0, :])
for color, col in zip(_colors, _cols):
    r, p, i = r_point[_key(col)], r_pval[_key(col)], _peak[col]
    ax.plot(age_grid, r, color=color, lw=0.9, alpha=0.22)                 # full trace, faded
    ax.plot(age_grid, np.where(p < ALPHA_SIG, r, np.nan),                 # bold where significant
            color=color, lw=1.8, alpha=1.0, label=col)
    ax.plot(age_grid[i], r[i], marker='o', color=color, mec='k', mew=0.5, ms=5, zorder=5)

# Age labels always go ABOVE their marker: a trough sits near the bottom of the axes, so labelling
# below it would land the number on the x tick labels. A label is pushed up an extra level only when
# an already-placed one is close in BOTH age and r -- markers far apart in r never need staggering
# even when their ages nearly coincide. Headroom is added first so a label above the highest peak
# is not clipped.
ax.set_ylim(top=ax.get_ylim()[1] + 0.10 * np.ptp(ax.get_ylim()))
_x_gap = 0.10 * np.ptp(age_grid)
_y_gap = 0.15 * np.ptp(ax.get_ylim())
_placed = {}                                    # level -> [(age, r) already labelled at that level]
for col in sorted(_cols, key=lambda c: age_grid[_peak[c]]):
    i, r = _peak[col], r_point[_key(col)]
    _x, _y, _lvl = age_grid[i], r[i], 0
    while any(abs(_x - px) < _x_gap and abs(_y - py) < _y_gap for px, py in _placed.get(_lvl, [])):
        _lvl += 1
    _placed.setdefault(_lvl, []).append((_x, _y))
    ax.annotate(f'{_x:.1f}', (_x, _y), textcoords='offset points', xytext=(0, 8 + 10 * _lvl),
                ha='center', fontsize=6, color='0.3')
ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
# The single stage boundary in use here. fig3 can draw several dashed lines when its n_bkps > 1,
# but those extra change points are annotation-only there and never define a stage.
ax.axvline(inflection_age, color='k', lw=0.8, ls='--', alpha=0.6)
ax.set_xlabel('Age (years)', fontsize=8)
ax.set_ylabel(f'Pearson $r$\n(map × {_ord_label[DERIV_ORDER_SHOW]} deriv.)', fontsize=8)
ax.tick_params(labelsize=7, pad=2)
ax.legend(fontsize=6, frameon=False, loc='best', ncol=1)

# ── Right row 2: the scatter behind each marker ──
_axz0 = None
for j, (color, col) in enumerate(zip(_colors, _cols)):
    axz   = fig.add_subplot(gs_coupling[1, j], sharey=_axz0 if ZOOM_SHARE_Y else None)
    _axz0 = _axz0 or axz
    i    = _peak[col]
    _map = neuro_families[PANEL_FAMILY][col].astype(float).values
    _y   = _deriv[:, i]
    _fin = np.isfinite(_map) & np.isfinite(_y)
    plot_correlation(pd.Series(_map[_fin]), pd.Series(_y[_fin]), axz, x_label=col, color=color,
                     y_label=f'GAM {_ord_label[DERIV_ORDER_SHOW]} deriv.', method='pearson',
                     show_stats=True, fontsize=8,
                     custom_pvalue=max(r_pval[_key(col)][i], 1 / N_PERMS))
    if j:
        axz.set_ylabel('')      # label the leftmost column only; plot_correlation substitutes its
                                # own default for a blank y_label, so clear it after the call
        if ZOOM_SHARE_Y:
            axz.tick_params(labelleft=False)   # shared scale -> inner tick labels are redundant,
                                               # and at this width they overran the panel to the left
    axz.set_title(f'{age_grid[i]:.1f} yr', fontsize=6, pad=0)
    axz.xaxis.set_major_locator(plt.MaxNLocator(TICK_MAX['x']))
    axz.yaxis.set_major_locator(plt.MaxNLocator(TICK_MAX['y']))
    axz.tick_params(labelsize=7, pad=2)
    print(f'  {col}: strongest coupling at {age_grid[i]:.1f} yr -- r={r_point[_key(col)][i]:+.3f}, '
          f'p_smash={max(r_pval[_key(col)][i], 1 / N_PERMS):.4f}'
          + ('*' if r_pval[_key(col)][i] < ALPHA_SIG else '') + f', n={int(_fin.sum())} parcels')

# ── Left block: one loading column per PC ──
_cmap = plt.get_cmap(PC_HEATMAP_CMAP)
for j, (pc, idx) in enumerate(zip(pc_names, pc_idx)):
    axp = fig.add_subplot(gs_pc[0, j])
    w   = pc_loadings[:, idx]
    # Stable subset first, top-N from it second. Walking the loading ranking and keeping the first
    # PC_TOP_N eligible genes is exactly "filter, then take the most extreme N of what survives".
    # Eligibility is pole-specific: the negative pole is judged on pc_neg_freq, the positive on pc_pos_freq.
    _rank_neg = np.argsort(w)                                       # most negative first
    _rank_pos = _rank_neg[::-1]                                     # most positive first
    _sel_neg  = [i for i in _rank_neg if pc_neg_freq[i, idx] >= PC_MIN_STABILITY][:PC_TOP_N]
    _sel_pos  = [i for i in _rank_pos if pc_pos_freq[i, idx] >= PC_MIN_STABILITY][:PC_TOP_N]
    rows = ([(i, pc_neg_freq[i, idx]) for i in _sel_neg]            # bottom: most negative
            + [(i, pc_pos_freq[i, idx]) for i in _sel_pos[::-1]])   # top: most positive
    _n_neg = len(_sel_neg)
    _vals  = [w[i] for i, _ in rows]
    _names = [pc_features[i] for i, _ in rows]

    # vmax comes from the FULL loading vector, not the plotted selection. The top-N per pole sit in
    # the top ~10-30% of a component's range (they are the flat head of a long ranking: PC1's
    # positive top-20 spans 0.003 of a 0.049 maximum), so normalising to the selection would stretch
    # that sliver across the whole colour range and invent within-pole structure the loadings do not
    # support. Normalised honestly the cells come out near saturated: colour reads as POLE, and the
    # printed value carries magnitude.
    _vmax = float(np.nanmax(np.abs(w)))
    _norm = plt.Normalize(vmin=-_vmax, vmax=_vmax)
    axp.imshow(np.asarray(_vals)[:, None], cmap=_cmap, norm=_norm, aspect='auto', origin='lower',
               extent=[-0.5, 0.5, -0.5, len(rows) - 0.5], interpolation='nearest')
    for p, (v, nm) in enumerate(zip(_vals, _names)):
        # Ink colour follows the cell it sits on, so saturated poles stay readable.
        _r, _g, _b, _ = _cmap(_norm(v))
        _ink = 'k' if (0.299 * _r + 0.587 * _g + 0.114 * _b) > 0.55 else 'w'
        axp.text(-0.45, p, nm, ha='left', va='center', fontsize=PC_CELL_FONTSIZE, color=_ink)
        axp.text(0.45, p, f'{v:+.3f}', ha='right', va='center', fontsize=PC_CELL_FONTSIZE,
                 color=_ink)
    if 0 < _n_neg < len(rows):
        axp.axhline(_n_neg - 0.5, color='k', lw=0.8)                # divider between the two poles
    axp.set_xticks([])
    axp.set_yticks([])
    axp.set_xlim(-0.5, 0.5)
    axp.set_ylim(-0.5, len(rows) - 0.5)
    for _sp in axp.spines.values():
        _sp.set_linewidth(0.5)
    if j == 0:
        axp.set_ylabel('PC loadings', fontsize=8)
    axp.set_title(f'{pc} ({pc_evr[idx]:.1%})', fontsize=8, pad=3,
                #   color='k')
                  color=_colors[j] if _linked else 'k')

    _st = np.array([st for _, st in rows])
    print(f'  {pc}: {len(rows)} genes shown ({_n_neg} negative pole, {len(rows) - _n_neg} positive); '
          f'stability median {np.median(_st):.0%}, range {_st.min():.0%}-{_st.max():.0%}')
    if PC_MIN_STABILITY > 0:
        # Report what the filter changed, so a hidden or promoted gene is never a silent edit.
        _raw  = list(_rank_neg[:PC_TOP_N]) + list(_rank_pos[:PC_TOP_N])
        _kept = set(_sel_neg) | set(_sel_pos)
        _hid  = [str(pc_features[i]) for i in _raw if i not in _kept]
        _new  = [str(pc_features[i]) for i in _sel_neg + _sel_pos if i not in set(_raw)]
        _pool = (int((pc_neg_freq[:, idx] >= PC_MIN_STABILITY).sum()),
                 int((pc_pos_freq[:, idx] >= PC_MIN_STABILITY).sum()))
        print(f'    stable pool: {_pool[0]} neg / {_pool[1]} pos genes clear '
              f'{PC_MIN_STABILITY:.0%} within the top {PC_STABILITY_WINDOW}')
        if len(_sel_neg) < PC_TOP_N or len(_sel_pos) < PC_TOP_N:
            print(f'    NOTE: pool smaller than PC_TOP_N={PC_TOP_N} at one or both poles '
                  f'({len(_sel_neg)} neg, {len(_sel_pos)} pos shown) -- raise PC_STABILITY_WINDOW '
                  f'or lower PC_MIN_STABILITY for equal-length panels.')
        print(f'    hidden (in the raw top-{PC_TOP_N} but unstable): {_hid or "none"}')
        print(f'    promoted (stable, from below the top-{PC_TOP_N}): {_new or "none"}')

# No tight_layout: it re-solves the nested grids and undoes BLOCK_WSPACE / GS_SPACING. The margins
# above are explicit instead, and save_panel's bbox_inches='tight' trims whatever sits outside.
_fam_tag = gene_tag if PANEL_FAMILY == 'Genes' else ''   # Myelin panels are gene-panel independent
_path = save_panel(fig, nas_scratch_dir, f'fig4_neurobio-pc_d{DERIV_ORDER_SHOW}_{PANEL_FAMILY}_{task_scan}'
                        f'_{signal_type}{variant_tag}{_fam_tag}')
print(f'{PANEL_FAMILY}: {len(_cols)} maps + {len(pc_names)} PC loading columns '
      f'-> {os.path.basename(_path)}.{{png,svg}}')
plt.show()

# ── PC x reference-map correlations (reported, not plotted) ──
_csv = f'{_path}_refcorr.csv'
pc_ref_table.to_csv(_csv)
print(f'\nPC x reference-map correlations ({PC_CORR_METHOD}, {N_PERMS} BrainSMASH surrogates of each '
      f'reference; * = p < {PC_ALPHA_SIG}. p is floored at 1/N_PERMS = {1 / N_PERMS:g}, the finest '
      f'a permutation test at this count can resolve).')
_fmt = pd.DataFrame({'var': pc_ref_table['var_explained'].map('{:.1%}'.format),
                     'n':   pc_ref_table['n_parcels']})
for _m in PC_REF_MAPS:
    _fmt[_m] = [f'{r:+.3f} (p={p:.3f})' + ('*' if p < PC_ALPHA_SIG else '')
                for r, p in zip(pc_ref_table[f'{_m}_r'], pc_ref_table[f'{_m}_p'])]
print(_fmt.to_string())
print(f'  table -> {os.path.basename(_csv)}')
